In [ ]:

#@title ✅ 一個儲存格示範 Qlib 能力（請在全新 Colab runtime 執行）

# 1. 抓原始碼（只為了拿範例的 YAML 設定檔）
!git clone -q https://github.com/microsoft/qlib.git /content/qlib

# 2. 安裝 Qlib 套件（會安裝到 site-packages，有編譯好的 _libs.rolling）
%cd /content/qlib
!pip install -q .

# 3. 很重要：離開原始碼目錄，避免 Python 優先載入 ./qlib 這個未編譯版本
%cd /content

# 4. 下載官方日頻示例資料（中國市場，來源 Yahoo Finance）
!python -m qlib.cli.data qlib_data \
    --target_dir ~/.qlib/qlib_data/cn_data \
    --region cn

# 5. 跑官方 LightGBM + Alpha158 workflow（完整：因子→模型→回測→評估）
!python -m qlib.cli.run \
    /content/qlib/examples/benchmarks/LightGBM/workflow_config_lightgbm_Alpha158.yaml

fatal: destination path '/content/qlib' already exists and is not an empty directory.
/content/qlib
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/content
2025-11-29 16:18:36.305 | WARNING  | qlib.tests.data:download:62 - The data for the example is collected from Yahoo Finance. Please be aware that the quality of the data might not be perfect. (You can refer to the original data source: https://finance.yahoo.com/lookup.)
2025-11-29 16:18:36.305 | INFO     | qlib.tests.data:download:65 - 20251129161836_qlib_data_cn_1d_latest.zip downloading......
196549632it [00:02, 66623433.67it/s]                   
2025-11-29 16:18:39.257 | WARNING  | qlib.tests.data:_unzip:124 - will delete the old qlib data directory(features, instruments, calendars, features_cache, dataset_cache): /root/.qlib/qlib_data/cn_data
2025-11-29 16:18:39.257 | INFO     | qlib.tests.data:_unzip:128 - /root/.qlib/qlib_data/cn_data/2025

In [ ]:

#@title 檢查 Qlib 實驗與 recorder 清單

import qlib
from qlib.workflow import R

# 1. 初始化（指向剛剛下載的 cn_data）
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region="cn")

# 2. 列出目前所有 experiment
exps = R.list_experiments()
print("現有 experiments：")
for e in exps:
    # 不同版本的 Experiment 物件屬性名稱可能略有差異，先印出 repr 即可
    print(" -", e)

# 3. 取得預設的 experiment（通常叫 workflow，如果你沒改 YAML）
exp = R.get_exp(experiment_name="workflow")

# 4. 列出這個 experiment 底下的 recorder（這裡回傳的是 dict）
recs = exp.list_recorders()
print("\nworkflow 底下的 recorders：")
for rec_id, rec in recs.items():
    print(f" - id={rec_id}, recorder={rec}")

[610:MainThread](2025-11-29 16:23:32,217) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[610:MainThread](2025-11-29 16:23:32,220) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[610:MainThread](2025-11-29 16:23:32,222) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/root/.qlib/qlib_data/cn_data')}


現有 experiments：
 - workflow
 - Default

workflow 底下的 recorders：
 - id=1d164bedc17648dfb4033c87050eef78, recorder={'class': 'Recorder', 'id': '1d164bedc17648dfb4033c87050eef78', 'name': 'mlflow_recorder', 'experiment_id': '681831588716335536', 'start_time': '2025-11-29 16:18:54', 'end_time': '2025-11-29 16:21:41', 'status': 'FAILED'}


In [ ]:

#@title 使用 Qlib 取得 CSI300 收盤價資料

import qlib
from qlib.data import D

# 1. 初始化 Qlib（使用剛下載的 cn_data）
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region="cn")

# 2. 定義市場與期間
market = "csi300"
start_time = "2017-01-01"
end_time   = "2020-08-01"
freq       = "day"

# 3. 先取得這個市場的股票清單（Qlib 需要 instruments 物件，而不是直接給字串）[web:112]
instruments = D.instruments(market=market)

# 4. 取得這些股票在區間內的收盤價
data = D.features(
    instruments=instruments,
    fields=["$close"],
    start_time=start_time,
    end_time=end_time,
    freq=freq,
)

data.head()

[610:MainThread](2025-11-29 16:30:11,598) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[610:MainThread](2025-11-29 16:30:11,604) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[610:MainThread](2025-11-29 16:30:11,605) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/root/.qlib/qlib_data/cn_data')}
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


$close
instrument datetime            
SH600000   2017-01-03  8.819036
           2017-01-04  8.835212
           2017-01-05  8.819036
           2017-01-06  8.754115
           2017-01-09  8.764877

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

#@title 以 5 日報酬當訊號，建立 TopK 策略並回測

import pandas as pd
from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.contrib.evaluate import backtest_daily

benchmark = "SH000300"  # Qlib 內建的滬深300指數作為基準

# 1. 從剛才的 data 取出收盤價，變成「日期 × 股票」矩陣
close = data["$close"].unstack(level=1)   # index=日期, columns=股票

# 2. 用 5 日報酬當作預測分數（只是示範，不代表好策略）
ret_5d = close.pct_change(5)
pred_score = ret_5d.stack().dropna()     # 變回 (日期, 股票) 的 Series，符合 TopkDropoutStrategy 需要的格式

# 3. 建立 TopK 策略物件（每天持有 50 檔，換掉最差的 5 檔）[web:139][web:154]
strategy_conf = {
    "topk": 50,
    "n_drop": 5,
    "signal": pred_score,
}
strategy = TopkDropoutStrategy(**strategy_conf)

# 4. 執行日頻回測
report, positions = backtest_daily(
    start_time=start_time,
    end_time=end_time,
    strategy=strategy,
    benchmark=benchmark,
)

report.head()

/tmp/ipython-input-1401095165.py:13: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret_5d = close.pct_change(5)
[610:MainThread](2025-11-29 16:30:57,715) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[610:MainThread](2025-11-29 16:30:57,716) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/c

backtest loop:   0%|          | 0/871 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.12/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.12/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/usr/local/lib/python3.12/dist-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.

,account,return,total_turnover,turnover,total_cost,cost,value,cash,bench
datetime,,,,,,,,,
2017-01-03,1.000000e+08,0.000000,0.0,0.000000,0.00000,0.000000,0.000000e+00,1.000000e+08,0.009713
2017-01-04,9.995725e+07,0.000000,85500000.0,0.855000,42750.00000,0.000427,8.550000e+07,1.445725e+07,0.007803
2017-01-05,1.000826e+08,0.001268,88246877.5,0.027481,44123.43875,0.000014,8.837359e+07,1.170900e+07,-0.000154
2017-01-06,9.957230e+07,-0.005099,88246877.5,0.000000,44123.43875,0.000000,8.786330e+07,1.170900e+07,-0.005974
2017-01-09,1.000510e+08,0.004808,88246877.5,0.000000,44123.43875,0.000000,8.834202e+07,1.170900e+07,0.004848


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:

#@title 對剛才的 TopK 策略做風險與報酬分析

import pandas as pd
from qlib.contrib.evaluate import risk_analysis

# 1. 確認 report 存在（來自上一個回測儲存格）
if "report" not in globals():
    raise RuntimeError("找不到 report，請先執行上一個回測儲存格。")

# 2. 計算「未扣成本」與「已扣成本」的超額報酬
analysis = {}
analysis["excess_return_without_cost"] = risk_analysis(
    report["return"] - report["bench"], freq="day"
)
analysis["excess_return_with_cost"] = risk_analysis(
    report["return"] - report["bench"] - report["cost"], freq="day"
)

# 3. 合併成一個 DataFrame 顯示
analysis_df = pd.concat(analysis)
analysis_df

risk
excess_return_without_cost mean               0.000302
                           std                0.004966
                           annualized_return  0.071795
                           information_ratio  0.937062
                           max_drawdown      -0.085413
excess_return_with_cost    mean               0.000277
                           std                0.004968
                           annualized_return  0.065914
                           information_ratio  0.860098
                           max_drawdown      -0.091759

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
